# LangChain Agent Minimum Start Notebook
----
Aim to download and load embedder and LM through HuggingFacePipeline and LangChain, and build a basic code sample for agent mode (hopefully add tools to keep constraints for structural output)

In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install GPUtil
%pip install -qU langchain langchain-huggingface transformers accelerate bitsandbytes
%pip install -qU "langchain[huggingface]"
%pip install huggingface_hub[hf_xet]
%pip install python-dotenv
%pip install ipywidgets
%pip install -qU langchain-ollama
%pip install -qU ollama

# Requires OLlama


Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import gc
import os
import getpass
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface.llms import HuggingFacePipeline
import transformers
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from transformers import AutoTokenizer
import logging
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_ollama import ChatOllama


def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

clear_vram()

In [3]:
clear_vram()
torch.cuda.is_available()

True

In [7]:
# ================================  LLM Configurations ================================

from langchain_anthropic import ChatAnthropic
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, pipeline


RUN_MODE = "api" # "api" or local_ollama     depreated => "local" / "local_quant"
# LOCAL_MODEL_ID = "Qwen/Qwen2.5-Coder-7B"
LOCAL_MODEL_ID = "Qwen/Qwen3-4B"
LOCAL_QUANT_MODEL_ID = "Qwen/Qwen3-32B"
API_MODEL_ID = "claude-sonnet-4-6"

OLLAMA_MODEL_ID = "qwen3:30B"

EMBED_MODEL_ID_OLLAMA = "qwen3-embedding:latest"
EMBED_MODEL_ID_HF = "Qwen/Qwen3-Embedding-8B"


# ================================  Load environment variables ================================
os.environ["LANGSMITH_TRACING"] = "true"


# Please include all API keys in the .env file in the root directory, if not, it will prompt for input here.

load_dotenv()

if RUN_MODE == "api" and not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your Hugging Face Token: ")
    
if not os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API Key: ")
if not os.getenv("LANGSMITH_PROJECT"):
    os.environ["LANGSMITH_PROJECT"] = getpass.getpass("Enter your LangSmith Project Name: ")



    

# --- LLM Factory Function ---

# I'm not deleting the code, but I'm switching to Ollama for local models.
# As this, please use local-ollama for local model inference.
# Also please use model names in Ollama format.

def get_llm():
    if RUN_MODE == "local":
        
        llm = HuggingFacePipeline.from_model_id(
            model_id=LOCAL_MODEL_ID,
            task="text-generation",
            device=0,# 0 for GPU, -1 for CPU
            pipeline_kwargs={
                "max_new_tokens": 1024,
            },
        )
        return ChatHuggingFace(llm=llm)
    elif RUN_MODE == "local_quant":
        # llm = HuggingFacePipeline.from_model_id(
        #     model_id=LOCAL_QUANT_MODEL_ID,
        #     task="text-generation",
        #     device=0,# 0 for GPU, -1 for CPU
        #     pipeline_kwargs={
        #         "max_new_tokens": 1024,
        #         "quantization_config": BitsAndBytesConfig(
        #             load_in_4bit=True,
        #             bnb_4bit_use_double_quant=True,
        #             bnb_4bit_quant_type="nf4",
        #             # bnb_4bit_compute_dtype=torch.bfloat16
        #         )
        #     },
        # )
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        # Manually load model and tokenizer with quantization config, to avoid parameter error when inferring
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_QUANT_MODEL_ID, trust_remote_code=True)
        print(tokenizer.chat_template)
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_QUANT_MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="sdpa" # accelerate attention implementation
        )

        pipeline = transformers.pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=1024,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        hf_llm = HuggingFacePipeline(pipeline=pipeline)
        return ChatHuggingFace(llm=hf_llm,tokenizer=tokenizer)
    
    elif RUN_MODE == "local_ollama":
        llm = ChatOllama(
        model=OLLAMA_MODEL_ID,
    #   format = JSON_SCHEMA
        temperature=0,
        num_ctx=8192,        
        num_predict=4096,
    )
        return llm
        

    elif RUN_MODE == "api":
        print(f"Connecting to Claude API for {API_MODEL_ID}...")
        # This uses Hugging Face's hosted infrastructure
        endpoint = ChatAnthropic(model=API_MODEL_ID) # type: ignore
        return endpoint
    else:
        print("Invalid RUN_MODE. Falling back to API mode.")
        endpoint = HuggingFaceEndpoint(
            repo_id=API_MODEL_ID,
            task="conversational",
            max_new_tokens=1024,
        
        )   
        return ChatHuggingFace(llm=endpoint)

        

In [8]:
# Loading Model

llm = get_llm()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert software engineer. All your responses MUST be in English."),
    ("user", "{input}")
])



Connecting to Claude API for claude-sonnet-4-6...


In [9]:
chain = prompt | llm | StrOutputParser() # type: ignore

test_query = "Hi?"
    
# Test Output
logging.basicConfig(level=logging.ERROR, force=True)
logging.getLogger("langchain").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("langchain_community").setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)
print(f"\n[Mode: {RUN_MODE}] Generating response...\n")
try:
    for chunk in chain.stream({"input": test_query}):
        print(chunk, end="", flush=True)
except Exception as e:
    print(f"An error occurred: {e}")


[Mode: api] Generating response...

Hi! How are you? Is there something I can help you with today? 😊

## Agent Wrapper with Basic Tools
---
To be extended

In [10]:
# =========================== TOOLS ===========================
import GPUtil as gutil
import platform
from langchain_core.utils.function_calling import convert_to_openai_tool
# Giving some basic tools to test.
@tool
def get_gpu_info(query: str = "") ->str:
    """ Retrieves information about the GPU(s) installed on the local machine.
        Call this whenever the user asks about their graphics card or GPU.
        Args:
        query: A string query describing what specific GPU info is needed. Defaults to empty.
    """
    gpus = gutil.getGPUs()
    if not gpus:
        return "No GPU found."
    gpu_info = []
    for gpu in gpus:
        gpu_info.append({
            "gpu_name": gpu.name,
            "memory_total": f"{gpu.memoryTotal} MB",
            "memory_used": f"{gpu.memoryUsed} MB",
            "memory_free": f"{gpu.memoryFree} MB",
            "temperature": f"{gpu.temperature} C",
            "CUDA availability": str(torch.cuda.is_available())
        })
        
    return str(gpu_info)
@tool
def get_cpu_info(query: str = "") -> str:
    """ Retrieves information about the CPU installed on the local machine.
        Call this whenever the user asks about their CPU.
        Args:
        query: A string query describing what specific CPU info is needed. Defaults to empty.
    """
    
    cpu_info = {
        "processor": platform.processor(),
        "architecture": platform.architecture(),
        "machine": platform.machine(),
        "platform": platform.platform(),
        "system": platform.system(),
        "version": platform.version(),
    }
    return str(cpu_info)

tools = [get_gpu_info, get_cpu_info]

get_gpu_info_schema = convert_to_openai_tool(get_gpu_info)
get_cpu_info_schema = convert_to_openai_tool(get_cpu_info)

In [11]:
from langchain.messages import HumanMessage
# ========================== AGENT DEMO ===========================
agent = create_agent(llm, tools=tools, system_prompt= """You are a helpful assistant. You have full access and are required to use tools to get graphic card or gpu information, along with CPU info.""")

msg = {'messages':[
    HumanMessage(content="Is cpu and gpu on my computer suitable for heavy gaming and AI inference?")
]}




agent.invoke(msg)

{'messages': [HumanMessage(content='Is cpu and gpu on my computer suitable for heavy gaming and AI inference?', additional_kwargs={}, response_metadata={}, id='201dfc01-72b9-4539-b41c-253662f45dc8'),
  AIMessage(content=[{'text': 'Sure! Let me fetch your CPU and GPU information at the same time right away!', 'type': 'text'}, {'id': 'toolu_0176RfRcw7ZJ6Vze73h1EfT2', 'caller': {'type': 'direct'}, 'input': {'query': 'CPU specs for heavy gaming and AI inference'}, 'name': 'get_cpu_info', 'type': 'tool_use'}, {'id': 'toolu_01Jfu6fyZztftG5D6yFL96D3', 'caller': {'type': 'direct'}, 'input': {'query': 'GPU specs for heavy gaming and AI inference'}, 'name': 'get_gpu_info', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01BfUWxXD6HeqT5UtV5qfbt9', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 

In [10]:

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "What's the temperature of the GPU of my computer?"}]})
    print(result)
except Exception as e:
    print(f"An error occurred: {e}")

{'messages': [HumanMessage(content="What's the temperature of the GPU of my computer?", additional_kwargs={}, response_metadata={}, id='29e0a500-b9ec-42b7-af05-c29a0fa7aeed'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:30B', 'created_at': '2026-02-03T02:37:30.0189628Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1514325900, 'load_duration': 98171700, 'prompt_eval_count': 285, 'prompt_eval_duration': 58082600, 'eval_count': 124, 'eval_duration': 1332230400, 'logprobs': None, 'model_name': 'qwen3:30B', 'model_provider': 'ollama'}, id='lc_run--019c215c-b037-7422-8db3-c0a10e332484-0', tool_calls=[{'name': 'get_gpu_info', 'args': {'query': 'temperature'}, 'id': '2cf60bc4-109c-4066-9666-0e336f7240d8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_tokens': 124, 'total_tokens': 409}), ToolMessage(content="[{'gpu_name': 'NVIDIA GeForce RTX 5090 Laptop GPU', 'memory_total': '24463.0 MB', 'memory_used': '

## RAG Agents

Using documents loader and embedder to enable models to search in embedded files. Can be used with libraries like FAISS

In [11]:
%pip install langchain langchain-text-splitters langchain-community bs4
%pip install langchain-community pypdf
%pip install -qU langchain-community faiss-cpu


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
## Exmaple of loding Document
from langchain_community.document_loaders import PyPDFLoader
import pprint

file_path = "data/nvda-form-10k.pdf"
loader = PyPDFLoader(file_path
                # ,mode="single",
                # this will change into single document mode, no split
                )

docs = loader.load() # split to page

print(len(docs))


pprint.pp(docs[0].metadata)

48
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0',
 'creator': 'EDGAR Filing HTML Converter',
 'creationdate': '2025-11-19T16:36:56-05:00',
 'title': '0001045810-25-000230',
 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group',
 'subject': 'Form 10-Q filed on 2025-11-19 for the period ending 2025-10-26',
 'keywords': '0001045810-25-000230; ; 10-Q',
 'moddate': '2025-11-19T16:37:07-05:00',
 'source': 'data/nvda-form-10k.pdf',
 'total_pages': 48,
 'page': 0,
 'page_label': '1'}


In [13]:
loader2 = PyPDFLoader(
    "data/nvda-form-10k.pdf",
    mode="single",
)
docs2 = loader2.load()
print(len(docs2))
pprint.pp(docs2[0].metadata)

1
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0',
 'creator': 'EDGAR Filing HTML Converter',
 'creationdate': '2025-11-19T16:36:56-05:00',
 'title': '0001045810-25-000230',
 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group',
 'subject': 'Form 10-Q filed on 2025-11-19 for the period ending 2025-10-26',
 'keywords': '0001045810-25-000230; ; 10-Q',
 'moddate': '2025-11-19T16:37:07-05:00',
 'source': 'data/nvda-form-10k.pdf',
 'total_pages': 48}


In [14]:
# Decoupled method of loading and parsing documents
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import PyPDFParser

loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path="data/",
        glob="nvda-form-10k.pdf",
    ),
    blob_parser=PyPDFParser(),
)
# An example of loading from web page, check out other loaders if needed!
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")
#         )
#     ),
# )
docs = loader.load()
print(docs[0].page_content)
pprint.pp(docs[0].metadata)

UNITED STATESSECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-Q
☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period ended October 26, 2025
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
Commission File Number: 0-23985
NVIDIA CORPORATION
(Exact name of registrant as specified in its charter)
Delaware 94-3177549
(State or other jurisdiction of (I.R.S. Employer
incorporation or organization) Identification No.)
2788 San Tomas Expressway, Santa Clara, California 95051
(Address of principal executive offices) (Zip Code)
(408) 486-2000(Registrant's telephone number, including area code)
N/A(Former name, former address and former fiscal year, if changed since last report)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class Trading Symbol(s) Name of each exchange on which registered
Common Stock, $0.001 par value per share NVDA The Nasdaq Gl

In [15]:
## Splitting: Pages may be too rought coarse for semantic search tasks.
# Text Spliter will separate documents into chunks with overlaps for better semantic search results.

from langchain_text_splitters import RecursiveCharacterTextSplitter
# Other splitters: https://docs.langchain.com/oss/python/integrations/splitters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

218


In [16]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpointEmbeddings
from langchain_ollama import OllamaEmbeddings
if RUN_MODE == "local_ollama":
    embeddings = OllamaEmbeddings(model=EMBED_MODEL_ID_OLLAMA)
elif RUN_MODE == "api":
    embeddings = HuggingFaceEndpointEmbeddings(
        model=EMBED_MODEL_ID_HF,
        task="feature-extraction",  # feature-extraction for embeddings
    )
## Embedding and storing chunks in vector form
from langchain_ollama import OllamaEmbeddings

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)
assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])



Generated vectors of length 4096

[-0.0030022063, -0.0009977841, -0.03706206, -0.04799659, -0.010066347, 0.03241564, -0.011364341, 0.014526162, -0.00202027, -0.021220006]


In [17]:
# Vector stores with FAISS - init
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [18]:
all_splits[0]

Document(metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2025-11-19T16:36:56-05:00', 'title': '0001045810-25-000230', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-Q filed on 2025-11-19 for the period ending 2025-10-26', 'keywords': '0001045810-25-000230; ; 10-Q', 'moddate': '2025-11-19T16:37:07-05:00', 'source': 'data\\nvda-form-10k.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content="UNITED STATESSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-Q\n☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the quarterly period ended October 26, 2025\nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nCommission File Number: 0-23985\nNVIDIA CORPORATION\n(Exact name of registrant as specified in its charter)\nDelaware 94-3177549\n(State or othe

In [19]:
from uuid import uuid4


uuids = [str(uuid4()) for _ in range(len(all_splits))]
vector_store.add_documents(documents=all_splits, ids=uuids)


['cf37f7d5-81e4-43a3-9949-875e274d81fb',
 'b3b3532b-0ced-4c5f-8aa6-5450a4eae8a3',
 '2131a5e7-f700-4192-971f-3297f46e048b',
 'bc3d7d4a-2597-49de-b77b-c621fcdea9fd',
 '7a931066-0f07-45ae-b9c0-9e8067bf518b',
 'ac4af18b-8eae-4287-88c9-1171e58eae0c',
 'cd6d492b-c8b3-4356-88a1-554e104c81df',
 'abe76ada-3f07-4e4b-be48-067c32d4917f',
 'b1a02019-e98f-41cc-83b5-b84deed25ace',
 'fbb3e094-145a-45bd-ade3-adb03235fd89',
 '26f39edf-ea41-468a-8501-4033c1669451',
 '30b33daa-f77e-4836-a04c-487381755d4e',
 '9c5cf18c-82b6-4ed2-bca2-4b9c19600207',
 '8d0e4dc7-9313-4fdc-b0af-889327c57747',
 'a78f59bc-df92-4ac8-87d0-e4c106d0a83d',
 'f7a86f18-4a24-4b4a-8afa-783ae3da9e9f',
 '76ee6bca-2a9c-4e94-a87f-344265e7d758',
 '870c7c71-feac-4c62-8967-b0e51b2a2325',
 '484f82d7-fcfd-4c7c-b9ff-be6e869bc971',
 '82f06d67-de9f-4928-8289-38f9129e06bb',
 '8d18c6bc-0f2a-4d38-9e0f-facd11e17469',
 '1320c4f0-3987-4a05-a014-1f6e01a8c77e',
 '7ef6932f-0615-4d79-bd3e-d66e1da121b4',
 'ec933f6e-7550-41fe-b9d2-26c49e4d3a4f',
 'ef7026a1-7779-

In [20]:
[doc.page_content for doc in vector_store.get_by_ids(uuids[:3])]

["UNITED STATESSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-Q\n☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the quarterly period ended October 26, 2025\nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nCommission File Number: 0-23985\nNVIDIA CORPORATION\n(Exact name of registrant as specified in its charter)\nDelaware 94-3177549\n(State or other jurisdiction of (I.R.S. Employer\nincorporation or organization) Identification No.)\n2788 San Tomas Expressway, Santa Clara, California 95051\n(Address of principal executive offices) (Zip Code)\n(408) 486-2000(Registrant's telephone number, including area code)\nN/A(Former name, former address and former fiscal year, if changed since last report)\nSecurities registered pursuant to Section 12(b) of the Act:\nTitle of each class Trading Symbol(s) Name of each exchange on which registered",
 'Securities registered pursuant to S

In [21]:
# perform search
results = vector_store.similarity_search_with_score(
    "How many Blackwell cards sold last FY?",
    k=3,
    filter=None # Can filter from metadata fields
)
doc, score = results[0]

print(f"Score: {score}\n")
print(doc.page_content)

Score: 1.1015832424163818

Graphics revenue – The year over year increase in the third quarter and first nine months of fiscal year 2026 was driven by sales of our Blackwell architecture.
Reportable segment operating income – The year over year increase in Compute & Networking segment operating income in the third quarter of fiscal year 2026was driven by growth in revenue. The year over year increase in Compute & Networking segment operating income in the first nine months of fiscal year 2026was driven by the growth in revenue, partially offset by a $4.5 billion charge associated with H20 excess inventory and purchase obligations in the first quarter of
fiscal year 2026. The year over year increase in Graphics segment operating income in the third quarter and first nine months of fiscal year 2026 was driven bythe growth in revenue.
Concentration of Revenue


In [22]:
# Async Version
results = vector_store.similarity_search(
    "How many Blackwell cards sold last FY?"
)

print(results[0])

page_content='Graphics revenue – The year over year increase in the third quarter and first nine months of fiscal year 2026 was driven by sales of our Blackwell architecture.
Reportable segment operating income – The year over year increase in Compute & Networking segment operating income in the third quarter of fiscal year 2026was driven by growth in revenue. The year over year increase in Compute & Networking segment operating income in the first nine months of fiscal year 2026was driven by the growth in revenue, partially offset by a $4.5 billion charge associated with H20 excess inventory and purchase obligations in the first quarter of
fiscal year 2026. The year over year increase in Graphics segment operating income in the third quarter and first nine months of fiscal year 2026 was driven bythe growth in revenue.
Concentration of Revenue' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2025-11-19T16:36:56-05:

In [23]:
# =================== RAG DemoImplementation ===================
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs


In [24]:
RAGtool = [retrieve_context]
RAGprompt = (
    "You have access to a tool that retrieves context from an earning report (Form 10-K). "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, RAGtool, system_prompt=RAGprompt)
query = "Summarize the key points from the earning report."
result = agent.invoke({"messages": [{"role": "user", "content": query }]})
print(result)

{'messages': [HumanMessage(content='Summarize the key points from the earning report.', additional_kwargs={}, response_metadata={}, id='a6aef038-dbf7-4889-9d2b-9d6c40b5e0e0'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:30B', 'created_at': '2026-02-03T02:38:10.8731048Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9332185900, 'load_duration': 6554036900, 'prompt_eval_count': 172, 'prompt_eval_duration': 111072600, 'eval_count': 250, 'eval_duration': 2570925600, 'logprobs': None, 'model_name': 'qwen3:30B', 'model_provider': 'ollama'}, id='lc_run--019c215d-3144-7fb1-b01b-0c98a349b405-0', tool_calls=[{'name': 'retrieve_context', 'args': {'query': 'Summarize the key points from the earning report.'}, 'id': '89b5057b-7e61-47f2-8610-b42a956060bf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 250, 'total_tokens': 422}), ToolMessage(content="Source: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.

In [25]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Summarize the key points from the earning report.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (9f9503b1-e55d-42c0-9a45-3f90947620f2)
 Call ID: 9f9503b1-e55d-42c0-9a45-3f90947620f2
  Args:
    query: key points
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2025-11-19T16:36:56-05:00', 'title': '0001045810-25-000230', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-Q filed on 2025-11-19 for the period ending 2025-10-26', 'keywords': '0001045810-25-000230; ; 10-Q', 'moddate': '2025-11-19T16:37:07-05:00', 'source': 'data\\nvda-form-10k.pdf', 'total_pages': 48, 'page': 24, 'page_label': '25', 'start_index': 1726}
Content: we expect.

## RAG Chain vesus 2-step Chain

RAG shown above owns the ability of LLMs to choose whether to invoke a query on their demand. However if we need robust search by a forced query,
we no longer call the model in a loop, but instead make a single pass.

In [26]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )

    return system_message


agent = create_agent(llm, tools=[], middleware=[prompt_with_context])

In [27]:
# Suppress all debug logs
logging.basicConfig(level=logging.ERROR, force=True)
logging.getLogger("langchain").setLevel(logging.ERROR)
logging.getLogger("langchain_community").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Summarize the key points from the earning report.
================================== Ai Message ==================================

Here's a concise summary of the key points from NVIDIA's earnings report (Q3 FY2026 vs. Q3 FY2025):

### 📈 **Revenue Growth (Strong Acceleration)**
- **Total Revenue**:  
  - **Q3**: $57.0B (up **62% YoY** from $35.1B)  
  - **9 Months**: $147.8B (up **62% YoY** from $91.2B)  
- **Segment Breakdown**:  
  - **Compute & Networking**:  
    - Q3: $50.9B (+64% YoY) | 9 Months: $131.8B (+64% YoY)  
    - *Driven by "three platform shifts"*.  
  - **Graphics**:  
    - Q3: $6.1B (+51% YoY) | 9 Months: $16.0B (+45% YoY)  
    - *Driven by Blackwell architecture sales*.  

### 💰 **Operating Profitability (Robust Growth)**
- **Total Operating Income**:  
  - **Q3**: $38.3B (+62% YoY) | **9 Months**: $92.6B (+49% YoY)  
- **Segment Highlights**:  
  - **Compute & Networking**:  
    -

## Using Outlines to Constraints JSON output


In [28]:
# %pip install outlines
# %pip install "outlines[ollama]"
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [29]:
print(result)

{'messages': [HumanMessage(content='Summarize the key points from the earning report.', additional_kwargs={}, response_metadata={}, id='a6aef038-dbf7-4889-9d2b-9d6c40b5e0e0'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:30B', 'created_at': '2026-02-03T02:38:10.8731048Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9332185900, 'load_duration': 6554036900, 'prompt_eval_count': 172, 'prompt_eval_duration': 111072600, 'eval_count': 250, 'eval_duration': 2570925600, 'logprobs': None, 'model_name': 'qwen3:30B', 'model_provider': 'ollama'}, id='lc_run--019c215d-3144-7fb1-b01b-0c98a349b405-0', tool_calls=[{'name': 'retrieve_context', 'args': {'query': 'Summarize the key points from the earning report.'}, 'id': '89b5057b-7e61-47f2-8610-b42a956060bf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 250, 'total_tokens': 422}), ToolMessage(content="Source: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.

In [30]:
metamodel_reader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path="data/",
        glob="metamodel.pdf",
    ),
    blob_parser=PyPDFParser(),
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader

metamodel_loader = PyMuPDFLoader("data/metamodel.pdf")
# An example of loading from web page, check out other loaders if needed!
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")
#         )
#     ),
# )
metamodel_doc = metamodel_loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True)
metamodel_splits = splitter.split_documents(metamodel_doc)



embedding_dim = len(embeddings.embed_query("hello world"))
index_metamodel = faiss.IndexFlatL2(embedding_dim)

vector_store_metamodel = FAISS(
    embedding_function=embeddings,
    index=index_metamodel,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

uuids_doc = [str(uuid4()) for _ in range(len(metamodel_splits))]

vector_store_metamodel.add_documents(documents=metamodel_splits, ids=uuids_doc)



['008af057-9377-4683-b1f4-4fe88ffff898',
 '030f1206-0971-43aa-afe9-9769fb6be03c',
 '23810ef3-a57e-4ae6-bb8a-5b2c9b6a24f8',
 'bfc0d8c1-910f-4298-ba28-f99e32f7b3c9',
 'a3523c43-8e03-418d-b499-3b673b57ced4',
 '75efdea0-8510-4625-a389-e3bebf9113de',
 '220b35c8-4abb-4d88-b7ca-adc2922cba3c',
 '0362b368-a024-48b0-adf7-f5e9566bcd9a',
 'd980f592-dcc5-4388-8200-55cbf35c33af',
 'f33b3d22-8409-4d6b-8b68-2561384ac740',
 '56dd1dc3-6000-4252-b55b-572af2601b83',
 '8ceb88c1-6d64-454a-9428-db95399ca980',
 'd9eb3107-9ad1-477d-ae99-9c8d7c167b52',
 '3134f98c-b47e-4822-bd29-e172372ba16d',
 'acaaf8d4-4d71-4ade-8a12-a27c4a4d58c5',
 'e4694352-9691-40be-84ce-e798f834dede',
 'df9d48bd-9f2a-4ba7-943b-2b38d93fbbc1',
 '01c5a048-9413-402a-9686-b432868d2e75',
 '613692ae-ac04-41c4-931f-c7a1bd2a7fac',
 'd7e9f974-63df-4b46-9b51-98291884d170',
 '221a176c-6f97-4e0a-8a90-f8e0b41032bf',
 '19327d16-a4ba-4623-9d53-a9bfe8ae0e20',
 '06b749a6-5490-4cdf-8823-c31bfa306ae0',
 '0649b386-3446-4325-a111-093cdc7a24b1',
 '32df7ecf-4d99-

In [ ]:
from typing import Annotated, Generic, Literal, TypeVar

from annotated_types import Gt

from pydantic import BaseModel

import json

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs



with open("data/aas.json", "r") as f:
    aas_schema = json.load(f)

class BaseAAS(BaseModel):
    assetAdministrationShells:list[dict]
    submodels:list[dict]
    conceptDescriptions:list[dict]
    

aas_agent = create_agent(llm, tools=[retrieve_context],response_format=aas_schema) # type: ignore
query = "Give me a simple AAS example for manufacturing, including full AAS representation including Environment and submodel, also with ConceptDescription. \
    Give me output of serialized JSON format. Search the metamodel to find relevant information."
for step in aas_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Give me a simple AAS example for manufacturing, including full AAS representation including Environment and submodel, also with ConceptDescription.     Give me output of serialized JSON format. Search the metamodel to find relevant information.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (2518fa42-f7ff-4126-9cc2-ae584140afcd)
 Call ID: 2518fa42-f7ff-4126-9cc2-ae584140afcd
  Args:
    query: AAS example for manufacturing with Environment, submodel, and ConceptDescription in JSON format
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2025-11-19T16:36:56-05:00', 'title': '0001045810-25-000230', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject'

: 